# BrainDecoding — Broderick 2018 自然语音 EEG GPU 解包与多阶段训练工作流

本 Notebook 专用于 **Google Colab GPU 实例**（L4 / A100 / T4）训练 **Broderick et al. (2018)** 英文自然语音 128 导联头皮脑电数据集（`broderick2018`，`all-n19` 全部 19 名受试者，20 段录音按 `16 Train / 2 Val / 2 Test` 固定划分，`50 Hz` 滤波信号 + `t5-large` 第 12 层词向量）：

1. **秒级解包与本地高速盘映射**：直接从 Google Drive `/content/drive/MyDrive/BrainDecoding/archives/features_broderick2018.tar` 解包至 Colab 本地 SSD `/content/local_cache`，并通过 `/content/studies_links` 软链接桥接原始路径，确保 `--require-cache` 100% 命中。
2. **严格零超参重调（Zero Hyperparameter Retuning）**：直接复用 Pallier 2025 验证稳定的训练超参数与 Pre-Roll（`45.0s` 缓冲区，`[-12, -19, -27, -36]s`，`M=2`）因果去偏配方，仅切换模型首层为 128 通道 `biosemi128` 空间注意力拓扑；按 BSPC 审稿防御设计聚焦 **3 种子（`SEED = '0,1,2'`）词级检索与因果去偏闭环**，无需运行 Stage 3 LLM 序列解码。
3. **三种开箱即用运行模式（`RUN_MODE`）**：
   - `'full_bspc_suite'`（默认推荐）：按依赖顺序一键串行训练全部 **4 个 BSPC 主表模型**（`word_only` -> `context_baseline` -> `context_debiased` -> `prefix_revision`），已完成的 `(模型, 种子)` 自动 0 秒跳过，断线重连通过 `--resume auto` 从 `last.ckpt` 无缝续训；
   - `'single_config'`：单配置独立训练（内置 `AUTO_TRAIN_MISSING_STAGE1 = True` 防护，若选择阶段 2 配置时云盘尚无对应阶段 1 检查点，会自动先补齐 Stage 1）；
   - `'tier_ab_probe'`：单种子（`PROBE_SEED = '0'`）对比 `null_space`（档位 A）与 `amortized_timing`（档位 B），并在 `val` 集盲选胜者。

In [ ]:
# 1. 检查 GPU、挂载 Google Drive、解析 Broderick 2018 训练队列并配置环境变量
import json as _json_init
import os
import shlex
import shutil
import subprocess
from pathlib import Path

if shutil.which("nvidia-smi"):
    subprocess.check_call(["nvidia-smi"])
else:
    print("[提示] 当前运行时未检测到 nvidia-smi（训练仍要求 CUDA GPU）。")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("非 Colab 环境，跳过 drive.mount")

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = "https://github.com/wdongdong0708-jpg/BrainDecoding2.git"  # @param {type:"string"}
GIT_BRANCH = "Broderick-EEG"  # @param {type:"string"}
DRIVE_BASE_DIR = "/content/drive/MyDrive/BrainDecoding"  # @param {type:"string"}

DATASET = "broderick2018"
MODALITY = "broderick2018"  # 对应 Google Drive 归档：archives/features_broderick2018.tar

# 运行模式选择：
#   - "full_bspc_suite" : 按依赖顺序一键训练 BSPC 词级主表 4 模型 x 3 种子 (word -> context -> context_debiased -> prefix_revision)
#   - "single_config"   : 仅训练下方 CONFIG_FILE 指定的单个配置（若缺少 Stage 1 会自动先补训）
#   - "tier_ab_probe"   : 单种子 (PROBE_SEED='0') 串行训练档位 A (null_space) 与档位 B (amortized_timing) 并在 val 盲选胜出档位
RUN_MODE = "full_bspc_suite"  # @param ["full_bspc_suite", "single_config", "tier_ab_probe"]
SEED = "0,1,2"  # @param {type:"string"}
PROBE_SEED = "0"  # @param {type:"string"}
SUBSET_ARGS = "--subject all"  # @param {type:"string"}

# 4 个 BSPC 主表模型的标准 exp_tag（与 colab_gpu_evaluation_broderick.ipynb 默认完全对齐）：
WORD_EXP_TAG = "lr4e-4_bs256"  # @param {type:"string"}
CONTEXT_BASELINE_EXP_TAG = "lr4e-4_bs256"  # @param {type:"string"}
CONTEXT_DEBIASED_EXP_TAG = "siamese_debiased_context_preroll"  # @param {type:"string"}
PREFIX_REVISION_EXP_TAG = "causal_null_space_debiased_prefix_m2"  # @param {type:"string"}
PREFIX_REVISION_CONFIG_FILE = "configs/training/siamese_debiased/broderick2018_causal_null_space_prefix_revision.yaml"  # @param {type:"string"}

# 单配置模式参数（仅当 RUN_MODE == "single_config" 时生效；可选 YAML 一览）：
#   - "configs/training/broderick2018_word_encoder.yaml"                                      (阶段 1:   Word-Only CNN)
#   - "configs/training/broderick2018_context_transformer.yaml"                               (阶段 2.1: Full-Context 非去偏基线)
#   - "configs/training/broderick2018_rolling_revision.yaml"                                  (阶段 2.2: 局部窗口滚动修订)
#   - "configs/training/broderick2018_prefix_revision.yaml"                                   (阶段 2.3: 全历史因果前缀滚动修订)
#   - "configs/training/siamese_debiased/broderick2018_siamese_debiased_context.yaml"         (阶段 2.1: Pre-Roll 孪生去偏 Full-Context)
#   - "configs/training/siamese_debiased/broderick2018_siamese_debiased_prefix_revision.yaml" (阶段 2.3: 旧版孪生去偏 Prefix-Revision)
#   - "configs/training/siamese_debiased/broderick2018_causal_null_space_prefix_revision.yaml"(阶段 2.3 档位 A: Pre-Roll + 零空间正交 10.0 + 欧氏跨视野 0.45 + M=2)
#   - "configs/training/siamese_debiased/broderick2018_amortized_timing_prefix_revision.yaml" (阶段 2.3 档位 B: 档位 A + TemporalGeometryBiasHead eta=0.4 + M=2)
CONFIG_FILE = "configs/training/siamese_debiased/broderick2018_causal_null_space_prefix_revision.yaml"  # @param {type:"string"}
EXP_TAG = "auto"  # @param {type:"string"}
PROBE_TAG_SUFFIX = "m2"  # @param {type:"string"}

AUTO_TRAIN_MISSING_STAGE1 = True  # @param {type:"boolean"}
RUN_VAL_EVAL_AFTER_TRAIN = True  # @param {type:"boolean"}
AUTO_UNASSIGN_RUNTIME = True  # @param {type:"boolean"}

WINDOW_SECONDS = 1.0
WINDOW_START_OFFSET_SECONDS = None

# 可选去偏超参覆盖（默认 None 100% 沿用 YAML 原配超参）：
DEBIAS_MODE = None
SHIFT_MODE = None
PREROLL_SECONDS = None
ETA_0 = None
ETA_DELTA = None
LAMBDA_NULL = None
LAMBDA_DELTA = None
LAMBDA_DISTILL = None
LAMBDA_DELTA_NULL = None
BETA_DELTA_CM = None
SHIFTS_PER_STEP = None
# ========================================================

_VALID_RUN_MODES = ("full_bspc_suite", "single_config", "tier_ab_probe")
if RUN_MODE not in _VALID_RUN_MODES:
    raise ValueError(f"不支持的 RUN_MODE={RUN_MODE!r}，可选：{_VALID_RUN_MODES}")

_active_seed_raw = PROBE_SEED if RUN_MODE == "tier_ab_probe" else SEED
if isinstance(_active_seed_raw, (list, tuple, set)):
    SEED_LIST = [int(s) for s in _active_seed_raw]
else:
    SEED_LIST = [
        int(s.strip())
        for s in str(_active_seed_raw).strip("[]()").replace(";", ",").replace(" ", ",").split(",")
        if s.strip()
    ]
SEED_ARG = ",".join(str(s) for s in SEED_LIST)

WINDOW_CLI_ARGS = []
if WINDOW_SECONDS is not None and str(WINDOW_SECONDS).strip() != "":
    WINDOW_CLI_ARGS.extend(["--window-seconds", str(WINDOW_SECONDS)])
if WINDOW_START_OFFSET_SECONDS is not None and str(WINDOW_START_OFFSET_SECONDS).strip() != "":
    WINDOW_CLI_ARGS.extend(["--window-start-offset-seconds", str(WINDOW_START_OFFSET_SECONDS)])
SUBSET_CLI_ARGS = shlex.split(str(SUBSET_ARGS or "").strip())


def _resolve_subset_glob(_subset_args: str) -> str:
    _txt = " ".join(str(_subset_args or "").strip().split())
    if not _txt or _txt in ("--subject all", "--subjects all"):
        return "all-*"
    _toks = _txt.split()
    if "--num-subjects" in _toks:
        _idx = _toks.index("--num-subjects")
        if _idx + 1 < len(_toks) and _toks[_idx + 1].isdigit():
            _n = int(_toks[_idx + 1])
            return "sub-01" if _n == 1 else f"subset-n{_n:02d}-*"
    return "*"


SUBSET_GLOB = _resolve_subset_glob(SUBSET_ARGS)

CODE_DIR = "/content/code"
VENV_DIR = "/content/proj-venv"
PYTHON = f"{VENV_DIR}/bin/python"
LINKS_DIR = "/content/studies_links"
LOCAL_CACHE_DIR = "/content/local_cache"

DRIVE_DATA_DIR = f"{DRIVE_BASE_DIR}/data"
DRIVE_ARCHIVE_DIR = f"{DRIVE_BASE_DIR}/archives"
DRIVE_RESULTS_DIR = f"{DRIVE_BASE_DIR}/runs"
DRIVE_MODELS_DIR = f"{DRIVE_BASE_DIR}/models"

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)


def _resolve_safe_probe_suffix(_ds: str, _raw_suffix: str) -> str:
    _suf = str(_raw_suffix or "m2").strip().strip("_") or "m2"
    _ds_root = Path(DRIVE_RESULTS_DIR) / _ds
    if not _ds_root.exists():
        return _suf
    _conflict = False
    for _prefix in ("causal_null_space_debiased_prefix", "amortized_timing_debiased_prefix"):
        _cand_dir = _ds_root / f"{_prefix}_{_suf}" / "prefix_revision"
        if not _cand_dir.exists():
            continue
        for _sum_p in _cand_dir.glob(f"{SUBSET_GLOB}/seed-*/training_summary.json"):
            try:
                _sdata = _json_init.loads(_sum_p.read_text(encoding="utf-8"))
                _siam = _sdata.get("siamese_debiased") or {}
                _ld = float(_siam.get("lambda_delta", -1.0))
                _ld_null = float(_siam.get("lambda_delta_null", 0.0))
                _m_step = int(_siam.get("shifts_per_step", 0))
                if abs(_ld - 0.45) > 1e-4 or _ld_null > 1e-6 or _m_step != 2:
                    _conflict = True
                    break
            except Exception:
                pass
    if _conflict:
        _alt = f"{_suf}_ld045"
        print(f"[防覆盖保护] 检测到云盘已有不同超参的 *_{_suf}，自动隔离至新后缀 *_{_alt}！")
        return _alt
    return _suf


ACTIVE_PROBE_SUFFIX = _resolve_safe_probe_suffix(DATASET, PROBE_TAG_SUFFIX)


def _infer_stage_from_config(_cfg_path: str) -> str:
    _low = str(_cfg_path).lower()
    if "word_encoder" in _low or "/main_word" in _low:
        return "word"
    if "prefix_revision" in _low:
        return "prefix_revision"
    if "context" in _low:
        return "context"
    return "revision"


def _resolve_auto_exp_tag(_cfg_path: str, _raw_tag: str) -> str:
    _clean_tag = str(_raw_tag or "").strip()
    if _clean_tag.lower() != "auto":
        return _clean_tag
    _low = str(_cfg_path).lower()
    if "word_encoder" in _low or "/main_word" in _low:
        return WORD_EXP_TAG
    if "siamese_debiased_context" in _low:
        return CONTEXT_DEBIASED_EXP_TAG
    if "context_transformer" in _low:
        return CONTEXT_BASELINE_EXP_TAG
    if "causal_null_space" in _low:
        return PREFIX_REVISION_EXP_TAG
    if "amortized_timing" in _low:
        return f"amortized_timing_debiased_prefix_{ACTIVE_PROBE_SUFFIX}"
    return Path(_cfg_path).stem


def _build_siamese_cli_args(_cfg_path: str, *, allow_debias_mode: bool = True) -> list[str]:
    _low = str(_cfg_path).lower()
    if "word_encoder" in _low or "/main_word" in _low:
        return []
    _is_rev = "revision" in _low
    _parts = []
    if allow_debias_mode and DEBIAS_MODE is not None and str(DEBIAS_MODE).strip() != "":
        _parts.extend(["--debias-mode", str(DEBIAS_MODE)])
    if SHIFT_MODE is not None and str(SHIFT_MODE).strip() != "":
        _parts.extend(["--shift-mode", str(SHIFT_MODE)])
    if PREROLL_SECONDS is not None and str(PREROLL_SECONDS).strip() != "":
        _parts.extend(["--preroll-seconds", str(PREROLL_SECONDS)])
    if allow_debias_mode and ETA_0 is not None and str(ETA_0).strip() != "":
        _parts.extend(["--eta-0", str(ETA_0)])
    if _is_rev and ETA_DELTA is not None and str(ETA_DELTA).strip() != "":
        _parts.extend(["--eta-delta", str(ETA_DELTA)])
    if LAMBDA_NULL is not None and str(LAMBDA_NULL).strip() != "":
        _parts.extend(["--lambda-null", str(LAMBDA_NULL)])
    if _is_rev and LAMBDA_DELTA is not None and str(LAMBDA_DELTA).strip() != "":
        _parts.extend(["--lambda-delta", str(LAMBDA_DELTA)])
    if LAMBDA_DISTILL is not None and str(LAMBDA_DISTILL).strip() != "":
        _parts.extend(["--lambda-distill", str(LAMBDA_DISTILL)])
    if _is_rev and LAMBDA_DELTA_NULL is not None and str(LAMBDA_DELTA_NULL).strip() != "":
        _parts.extend(["--lambda-delta-null", str(LAMBDA_DELTA_NULL)])
    if _is_rev and BETA_DELTA_CM is not None and str(BETA_DELTA_CM).strip() != "":
        _parts.extend(["--beta-delta-cm", str(BETA_DELTA_CM)])
    if SHIFTS_PER_STEP is not None and str(SHIFTS_PER_STEP).strip() != "":
        _parts.extend(["--shifts-per-step", str(SHIFTS_PER_STEP)])
    return _parts


_CAUSAL_CONTROL_CONFIG = "configs/controls/common_mode_subtraction/causal_common_mode_subtraction_control.yaml"

if RUN_MODE == "full_bspc_suite":
    _prefix_tag_resolved = (
        f"amortized_timing_debiased_prefix_{ACTIVE_PROBE_SUFFIX}"
        if "amortized_timing" in PREFIX_REVISION_CONFIG_FILE.lower()
        and PREFIX_REVISION_EXP_TAG == "causal_null_space_debiased_prefix_m2"
        else PREFIX_REVISION_EXP_TAG
    )
    TRAIN_JOBS = [
        {
            "slot": "word_only",
            "label": "Stage 1: Word-Only CNN",
            "short_label": "Word-Only",
            "dataset": DATASET,
            "stage": "word",
            "config_file": "configs/training/broderick2018_word_encoder.yaml",
            "exp_tag": WORD_EXP_TAG,
            "siamese_cli_args": [],
            "eval_control_config": _CAUSAL_CONTROL_CONFIG,
        },
        {
            "slot": "context_unfrozen",
            "label": "Stage 2.1: Full-Context 非去偏基线",
            "short_label": "Ctx-Base",
            "dataset": DATASET,
            "stage": "context",
            "config_file": "configs/training/broderick2018_context_transformer.yaml",
            "exp_tag": CONTEXT_BASELINE_EXP_TAG,
            "siamese_cli_args": _build_siamese_cli_args("configs/training/broderick2018_context_transformer.yaml"),
            "eval_control_config": _CAUSAL_CONTROL_CONFIG,
        },
        {
            "slot": "context_debiased",
            "label": "Stage 2.1: Full-Context Pre-Roll 孪生去偏",
            "short_label": "Ctx-Debiased",
            "dataset": DATASET,
            "stage": "context",
            "config_file": "configs/training/siamese_debiased/broderick2018_siamese_debiased_context.yaml",
            "exp_tag": CONTEXT_DEBIASED_EXP_TAG,
            "siamese_cli_args": _build_siamese_cli_args("configs/training/siamese_debiased/broderick2018_siamese_debiased_context.yaml"),
            "eval_control_config": _CAUSAL_CONTROL_CONFIG,
        },
        {
            "slot": "prefix_revision",
            "label": "Stage 2.3: Causal Null-Space Prefix-Revision",
            "short_label": "Prefix-Debiased",
            "dataset": DATASET,
            "stage": "prefix_revision",
            "config_file": PREFIX_REVISION_CONFIG_FILE,
            "exp_tag": _prefix_tag_resolved,
            "siamese_cli_args": _build_siamese_cli_args(PREFIX_REVISION_CONFIG_FILE),
            "eval_control_config": _CAUSAL_CONTROL_CONFIG,
        },
    ]
    EXP_TAG = _prefix_tag_resolved
elif RUN_MODE == "tier_ab_probe":
    _tier_a_cfg = "configs/training/siamese_debiased/broderick2018_causal_null_space_prefix_revision.yaml"
    _tier_b_cfg = "configs/training/siamese_debiased/broderick2018_amortized_timing_prefix_revision.yaml"
    TRAIN_JOBS = [
        {
            "slot": "tier_a",
            "label": "档位 A (Causal Null-Space | lambda_null=10, lambda_delta=0.45, M=2)",
            "short_label": "Tier A",
            "dataset": DATASET,
            "stage": "prefix_revision",
            "config_file": _tier_a_cfg,
            "exp_tag": f"causal_null_space_debiased_prefix_{ACTIVE_PROBE_SUFFIX}",
            "siamese_cli_args": _build_siamese_cli_args(_tier_a_cfg, allow_debias_mode=False),
            "eval_control_config": _CAUSAL_CONTROL_CONFIG,
        },
        {
            "slot": "tier_b",
            "label": "档位 B (Amortized Timing | Tier A + TemporalGeometryBiasHead eta=0.4, M=2)",
            "short_label": "Tier B",
            "dataset": DATASET,
            "stage": "prefix_revision",
            "config_file": _tier_b_cfg,
            "exp_tag": f"amortized_timing_debiased_prefix_{ACTIVE_PROBE_SUFFIX}",
            "siamese_cli_args": _build_siamese_cli_args(_tier_b_cfg, allow_debias_mode=False),
            "eval_control_config": _CAUSAL_CONTROL_CONFIG,
        },
    ]
    EXP_TAG = TRAIN_JOBS[0]["exp_tag"]
else:
    EXP_TAG = _resolve_auto_exp_tag(CONFIG_FILE, EXP_TAG)
    _single_stage = _infer_stage_from_config(CONFIG_FILE)
    TRAIN_JOBS = [
        {
            "slot": "single",
            "label": f"单配置训练 ({Path(CONFIG_FILE).stem})",
            "short_label": EXP_TAG or Path(CONFIG_FILE).stem,
            "dataset": DATASET,
            "stage": _single_stage,
            "config_file": CONFIG_FILE,
            "exp_tag": EXP_TAG,
            "siamese_cli_args": _build_siamese_cli_args(CONFIG_FILE),
            "eval_control_config": _CAUSAL_CONTROL_CONFIG,
        }
    ]

# 统一通过环境变量驱动所有路径（与 CPU 预热完全对齐）
os.environ["BRAINDECODING_DATA_DIR"] = LINKS_DIR
os.environ["BRAINDATA_ROOT"] = LINKS_DIR
os.environ["BRAINDECODING_CACHE_DIR"] = LOCAL_CACHE_DIR
os.environ["BRAINDECODING_ARCHIVE_DIR"] = DRIVE_ARCHIVE_DIR
os.environ["BRAINDECODING_RESULTS_DIR"] = DRIVE_RESULTS_DIR
os.environ["BRAINDECODING_EXP_TAG"] = EXP_TAG
os.environ["BRAINDECODING_WORD_EXP_TAG"] = WORD_EXP_TAG
os.environ["BRAINDECODING_MODEL_DIR"] = DRIVE_MODELS_DIR
os.environ["HF_HOME"] = f"{LOCAL_CACHE_DIR}/huggingface"
os.environ["PYTHONUNBUFFERED"] = "1"

print("DATASET                   =", DATASET)
print("GIT_BRANCH                =", GIT_BRANCH)
print("RUN_MODE                  =", RUN_MODE)
print("SEEDS                     =", SEED_LIST, f"(--seed {SEED_ARG})")
print("MODALITY / ARCHIVE        =", f"{MODALITY} -> features_{MODALITY}.tar")
print("WINDOW_SECONDS            =", WINDOW_SECONDS if WINDOW_SECONDS is not None else "(from YAML default 1.0s)")
print("BRAINDECODING_CACHE_DIR   =", os.environ["BRAINDECODING_CACHE_DIR"])
print("BRAINDECODING_RESULTS_DIR =", os.environ["BRAINDECODING_RESULTS_DIR"])
print("BRAINDECODING_WORD_EXP_TAG=", os.environ["BRAINDECODING_WORD_EXP_TAG"] or "(same as EXP_TAG)")
print("QUEUED TRAIN_JOBS         =", len(TRAIN_JOBS))
for _i, _j in enumerate(TRAIN_JOBS, 1):
    print(f"  [{_i}/{len(TRAIN_JOBS)}] {_j['label']} -> runs/{_j['dataset']}/{_j['exp_tag']}/{_j['stage']}/... (cfg={_j['config_file']})")

In [ ]:
# 2. 克隆或同步最新代码到 /content/code，初始化虚拟环境并建立 /content/studies_links 软链接桥
import subprocess
import sys
from pathlib import Path

code_path = Path(CODE_DIR)
if not (code_path / ".git").is_dir():
    subprocess.check_call(["git", "clone", "-b", GIT_BRANCH, GIT_REPO_URL, CODE_DIR])
else:
    subprocess.check_call(["git", "-C", CODE_DIR, "fetch", "origin"])
    subprocess.check_call(["git", "-C", CODE_DIR, "checkout", GIT_BRANCH])
    subprocess.check_call(["git", "-C", CODE_DIR, "pull", "--ff-only"])

subprocess.check_call(
    [
        sys.executable,
        "-u",
        str(code_path / "scripts/colab_setup.py"),
        "--venv-dir",
        VENV_DIR,
        "--project-root",
        CODE_DIR,
        "--drive-data-dir",
        DRIVE_DATA_DIR,
        "--links-dir",
        LINKS_DIR,
        "--local-cache-dir",
        LOCAL_CACHE_DIR,
    ]
)

In [ ]:
# 3. 从 Google Drive 秒级解包 archives/features_broderick2018.tar 到 /content/local_cache 并核验缓存
import os
import subprocess
from pathlib import Path

archive_file = Path(DRIVE_ARCHIVE_DIR) / f"features_{MODALITY}.tar"
if not archive_file.is_file():
    raise FileNotFoundError(
        f"Google Drive 中未找到 Broderick 2018 缓存归档：{archive_file}（期望 features_broderick2018.tar）。请先运行 colab_cpu_cache.ipynb。"
    )

print(f"[解包归档] {archive_file} ({archive_file.stat().st_size / (1024**3):.2f} GiB) -> {LOCAL_CACHE_DIR}")
subprocess.check_call(
    [
        PYTHON,
        "-u",
        "scripts/cache_archive.py",
        "unpack",
        "--archive-dir",
        DRIVE_ARCHIVE_DIR,
        "--cache-dir",
        LOCAL_CACHE_DIR,
        "--modality",
        MODALITY,
        "--skip-existing",
    ],
    cwd=CODE_DIR,
    env=os.environ.copy(),
)

ds_cache_dir = Path(LOCAL_CACHE_DIR) / "broderick2018"
required_artifacts = [
    ds_cache_dir / "events/events.csv",
    ds_cache_dir / "signals/manifest.json",
    ds_cache_dir / "text/t5_large_layer_0_5/embeddings.npz",
    ds_cache_dir / "events/manifests/vocabulary_N50.json",
]
for art in required_artifacts:
    if not art.is_file():
        raise FileNotFoundError(f"解包后缺少 Broderick 2018 核心缓存文件：{art}")
    print(f"[OK] {art.relative_to(Path(LOCAL_CACHE_DIR))} ({art.stat().st_size:,} bytes)")

In [ ]:
# 4. 按依赖顺序启动 GPU 训练（自动补齐缺失 Stage 1；已完成种子 0 秒跳过；断线重连自动从 last.ckpt 续训）
import json
import os
import subprocess
from pathlib import Path


def _is_seed_training_completed(_ds: str, _tag: str, _stage: str, _s: int) -> bool:
    _base = (Path(DRIVE_RESULTS_DIR) / _ds / _tag / _stage) if _tag else (Path(DRIVE_RESULTS_DIR) / _ds / _stage)
    if not _base.exists():
        return False
    for _summary_p in _base.glob(f"{SUBSET_GLOB}/seed-{int(_s):03d}/training_summary.json"):
        try:
            _meta = json.loads(_summary_p.read_text(encoding="utf-8"))
            if _meta.get("status") == "completed" and (
                (_summary_p.parent / "best.ckpt").is_file() or (_summary_p.parent / "best.pt").is_file()
            ):
                return True
        except Exception:
            pass
    return False


def _run_train_job(
    *,
    label: str,
    cfg_file: str,
    exp_tag: str,
    stage: str,
    seeds: list[int],
    siamese_cli_args: list[str],
) -> None:
    seed_str = ",".join(str(s) for s in seeds)
    cmd = [
        PYTHON,
        "-u",
        "-m",
        "braindecoding",
        "train",
        "--config",
        cfg_file,
        "--seed",
        seed_str,
        "--require-cache",
        "--resume",
        "auto",
    ]
    if exp_tag:
        cmd.extend(["--exp-tag", exp_tag])
    if WORD_EXP_TAG and stage != "word":
        cmd.extend(["--word-exp-tag", WORD_EXP_TAG])
    cmd.extend(WINDOW_CLI_ARGS)
    cmd.extend(siamese_cli_args)
    cmd.extend(SUBSET_CLI_ARGS)
    env = os.environ.copy()
    env["BRAINDECODING_EXP_TAG"] = exp_tag
    print(f"[执行命令] ({label}) {' '.join(cmd)}")
    subprocess.check_call(cmd, cwd=CODE_DIR, env=env)


for _job_idx, _job in enumerate(TRAIN_JOBS, 1):
    _job_cfg = _job["config_file"]
    _job_tag = _job["exp_tag"]
    _job_ds = _job["dataset"]
    _job_stage = _job["stage"]
    _pending_seeds = [_s for _s in SEED_LIST if not _is_seed_training_completed(_job_ds, _job_tag, _job_stage, _s)]
    print("\n" + "=" * 86)
    print(f"[训练队列 {_job_idx}/{len(TRAIN_JOBS)}] {_job['label']} | exp_tag={_job_tag} | seeds={SEED_LIST}")
    print("=" * 86)
    if not _pending_seeds:
        print(f"[0秒跳过] {_job['label']} 的全部种子 {SEED_LIST} 已完成训练并存在 best.ckpt，无需重训！")
        continue

    if _job_stage != "word" and AUTO_TRAIN_MISSING_STAGE1:
        _missing_s1 = [
            _s for _s in _pending_seeds if not _is_seed_training_completed(_job_ds, WORD_EXP_TAG, "word", _s)
        ]
        if _missing_s1:
            print(f"[自动前置] Stage 2 依赖的 Stage 1 ({WORD_EXP_TAG}) 缺少种子 {_missing_s1}，先自动训练 Stage 1...")
            _run_train_job(
                label="Stage 1: Word-Only CNN (自动前置补齐)",
                cfg_file="configs/training/broderick2018_word_encoder.yaml",
                exp_tag=WORD_EXP_TAG,
                stage="word",
                seeds=_missing_s1,
                siamese_cli_args=[],
            )

    _run_train_job(
        label=_job["label"],
        cfg_file=_job_cfg,
        exp_tag=_job_tag,
        stage=_job_stage,
        seeds=_pending_seeds,
        siamese_cli_args=_job["siamese_cli_args"],
    )

In [ ]:
# 5. 验证集 (val) 因果评估与总览看板（严格零测试集窥探；tier_ab_probe 模式自动导出 VAL 盲选对比）
import json
import os
import subprocess
from pathlib import Path


def _is_split_eval_fresh(_ds: str, _tag: str, _stage: str, _s: int, _split: str, *, expect_cms: bool = True) -> bool:
    _base = (Path(DRIVE_RESULTS_DIR) / _ds / _tag / _stage) if _tag else (Path(DRIVE_RESULTS_DIR) / _ds / _stage)
    if not _base.exists():
        return False
    _rel_target = (
        f"exploratory/common_mode_subtraction_{_split}.json"
        if expect_cms
        else f"evaluation_{_split}.json"
    )
    for _target_p in _base.glob(f"{SUBSET_GLOB}/seed-{int(_s):03d}/{_rel_target}"):
        _seed_dir = _target_p.parent.parent if expect_cms else _target_p.parent
        _ckpt_p = _seed_dir / "best.ckpt"
        if _target_p.is_file() and (not _ckpt_p.is_file() or _target_p.stat().st_mtime >= _ckpt_p.stat().st_mtime):
            return True
    return False


def _collect_val_rows(_job: dict) -> list[dict]:
    _rows = []
    _j_base = Path(DRIVE_RESULTS_DIR) / _job["dataset"] / _job["exp_tag"] / _job["stage"]
    for _s in SEED_LIST:
        _cms_matches = sorted(
            _j_base.glob(f"{SUBSET_GLOB}/seed-{int(_s):03d}/exploratory/common_mode_subtraction_val.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if _cms_matches:
            try:
                _cms_d = json.loads(_cms_matches[0].read_text(encoding="utf-8"))
                for _entry in (_cms_d.get("sweep") or []):
                    _eta = float(_entry.get("eta", 0.0))
                    _c_ref = _entry.get("clean_reference") or {}
                    _s_agg = _entry.get("shifted_aggregate") or {}
                    _c50 = float(_c_ref.get("vocab50_macro_top10", 0.0)) * 100.0
                    _s50 = float(_s_agg.get("mean_vocab50_macro_top10", _s_agg.get("shifted_mean_vocab50_macro_top10", 0.0))) * 100.0
                    _g50 = float(_s_agg.get("clean_minus_shifted_vocab50_macro_top10", (_c50 - _s50) / 100.0)) * 100.0
                    _t1_50 = float(_c_ref.get("vocab50_macro_top1", 0.0)) * 100.0
                    _c_full = float(_c_ref.get("full_vocab_word_macro_top10", _c_ref.get("full_vocab_macro_top10", 0.0))) * 100.0
                    _mic_full = float(_c_ref.get("full_vocab_micro_top10", 0.0)) * 100.0
                    _g_full = float(_s_agg.get("clean_minus_shifted_full_vocab_word_macro_top10", _s_agg.get("clean_minus_shifted_full_vocab_macro_top10", 0.0))) * 100.0
                    _mode_tag = f"{_job['short_label']} Raw" if abs(_eta) < 1e-9 else f"{_job['short_label']} EMA({_eta:g})"
                    _rows.append(
                        {
                            "seed": int(_s),
                            "label": _mode_tag,
                            "short_label": _job["short_label"],
                            "config_file": _job["config_file"],
                            "exp_tag": _job["exp_tag"],
                            "eta": _eta,
                            "c50": _c50,
                            "s50": _s50,
                            "g50": _g50,
                            "t1_50": _t1_50,
                            "c_full": _c_full,
                            "mic_full": _mic_full,
                            "g_full": _g_full,
                        }
                    )
                continue
            except Exception as _exc:
                print(f"[提示] 读取 {_cms_matches[0]} 失败: {_exc}")

        _eval_matches = sorted(
            _j_base.glob(f"{SUBSET_GLOB}/seed-{int(_s):03d}/evaluation_val.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if _eval_matches:
            try:
                _ev = json.loads(_eval_matches[0].read_text(encoding="utf-8"))
                _metrics = _ev.get("metrics") or {}
                _fv = _ev.get("full_vocabulary") or {}
                _c50 = float(_metrics.get("vocab50_macro_top10", 0.0)) * 100.0
                _t1_50 = float(_metrics.get("vocab50_macro_top1", 0.0)) * 100.0
                _c_full = float(_fv.get("macro_word_top10", _fv.get("macro_top10", 0.0))) * 100.0
                _mic_full = float(_fv.get("micro_top10", 0.0)) * 100.0
                _rows.append(
                    {
                        "seed": int(_s),
                        "label": f"{_job['short_label']} Clean",
                        "short_label": _job["short_label"],
                        "config_file": _job["config_file"],
                        "exp_tag": _job["exp_tag"],
                        "eta": 0.0,
                        "c50": _c50,
                        "s50": 0.0,
                        "g50": 0.0,
                        "t1_50": _t1_50,
                        "c_full": _c_full,
                        "mic_full": _mic_full,
                        "g_full": 0.0,
                    }
                )
            except Exception as _exc:
                print(f"[提示] 读取 {_eval_matches[0]} 失败: {_exc}")
    return _rows


if RUN_VAL_EVAL_AFTER_TRAIN:
    for _job_idx, _job in enumerate(TRAIN_JOBS, 1):
        _job_cfg = _job["config_file"]
        _job_tag = _job["exp_tag"]
        _job_ds = _job["dataset"]
        _job_stage = _job["stage"]
        _ctrl_cfg = str(_job.get("eval_control_config") or "").strip()
        _pending_val = [
            _s
            for _s in SEED_LIST
            if not _is_split_eval_fresh(_job_ds, _job_tag, _job_stage, _s, "val", expect_cms=bool(_ctrl_cfg))
        ]
        print("\n" + "=" * 86)
        print(f"[验证集评估 {_job_idx}/{len(TRAIN_JOBS)}] {_job['label']} | exp_tag={_job_tag}")
        print("=" * 86)
        if not _pending_val:
            print(f"[0秒跳过] {_job['label']} 的 VAL 评估已是最新，直接读取已落盘 JSON。")
            continue
        _val_cmd = [
            PYTHON,
            "-u",
            "-m",
            "braindecoding",
            "eval-run",
            "--config",
            _job_cfg,
            "--seed",
            ",".join(str(_s) for _s in _pending_val),
            "--split",
            "val",
            "--require-cache",
        ]
        if _job_tag:
            _val_cmd.extend(["--exp-tag", _job_tag])
        if WORD_EXP_TAG and _job_stage != "word":
            _val_cmd.extend(["--word-exp-tag", WORD_EXP_TAG])
        _val_cmd.extend(WINDOW_CLI_ARGS)
        if _ctrl_cfg:
            _val_cmd.extend(
                [
                    "--control-config",
                    _ctrl_cfg,
                    "--control-method",
                    "raw,causal_cms:eta=0.5",
                ]
            )
        _val_cmd.extend(SUBSET_CLI_ARGS)
        _env_val = os.environ.copy()
        _env_val["BRAINDECODING_EXP_TAG"] = _job_tag
        subprocess.check_call(_val_cmd, cwd=CODE_DIR, env=_env_val)

_all_val_rows = []
for _job in TRAIN_JOBS:
    _all_val_rows.extend(_collect_val_rows(_job))

if _all_val_rows:
    _lines = [
        "=" * 92,
        f"Broderick 2018 VAL Summary | Pre-Roll [-10..-40s] + 1x Online | seeds={SEED_LIST}",
        "=" * 92,
        f"{'Model / Mode (Seed)':<30} | {'50wCln':>6} | {'50wShf':>6} | {'50wGap':>7} | {'50wTp1':>6} | {'FulMac':>6} | {'FulMic':>6}",
        "-" * 92,
    ]
    for _r in _all_val_rows:
        _nm = f"{_r['label']} s{_r['seed']}"
        _lines.append(
            f"{_nm:<30} | {_r['c50']:>5.2f}% | {_r['s50']:>5.2f}% | {_r['g50']:>+6.2f}% | {_r['t1_50']:>5.2f}% | {_r['c_full']:>5.2f}% | {_r['mic_full']:>5.2f}%"
        )
    if RUN_MODE == "tier_ab_probe":
        _best_val_row = max(_all_val_rows, key=lambda r: (r["g50"], r["g_full"], r["c50"]))
        _lines.append("-" * 92)
        _lines.append(
            f" * VAL Blind Winner : {_best_val_row['label']} (VAL 50w Gap={_best_val_row['g50']:+.2f}%, FulMic={_best_val_row['mic_full']:.2f}%)"
        )
        _lines.append(
            f" * Winner YAML      : {Path(_best_val_row['config_file']).name} (exp_tag={_best_val_row['exp_tag']})"
        )
        _cmp_val_out = Path(DRIVE_RESULTS_DIR) / DATASET / f"probe_tier_ab_{ACTIVE_PROBE_SUFFIX}_comparison_val.md"
        _cmp_val_out.write_text("```text\n" + "\n".join(_lines) + "\n```\n", encoding="utf-8")
        print(f"[已保存 Tier A/B VAL 盲选看板] {_cmp_val_out}")
    _lines.append("=" * 92)
    print("\n" + "\n".join(_lines))

In [ ]:
# 6. 同步落盘并按 AUTO_UNASSIGN_RUNTIME 决定是否自动释放 GPU 运行时
import os
import time

if hasattr(os, "sync"):
    os.sync()

if AUTO_UNASSIGN_RUNTIME:
    try:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        print("数据已全部同步至 Google Drive，10 秒后释放运行时以停止扣费...")
        time.sleep(10)
        runtime.unassign()
    except ImportError:
        print("非 Colab 环境，跳过 drive.flush_and_unmount / runtime.unassign。")
else:
    print("AUTO_UNASSIGN_RUNTIME=False：已保留当前 Colab GPU 会话供交互查看。")